# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
from pandas import DataFrame
import plotly.express as px
import random

# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   str    
 1   continent  1704 non-null   str    
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   str    
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), str(3)
memory usage: 106.6 KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: <StringArray>
['Asia', 'Europe', 'Africa', 'Americas', 'Oceania']
Length: 5, dtype: str
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2000 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [3]:
# Task 1
# YOUR CODE HERE

Continent = random.choice(list(df["continent"].drop_duplicates()))
Continent = "Asia"
Years = [2002,2007]
df1 = df.loc[df['continent'] == Continent]
df2 = df1[df1["year"].isin(Years)]
df2["year"] = df2["year"].astype("str")



fig = px.scatter(
    data_frame=df2, x='gdpPercap', y='lifeExp',
    color='year',                     
    hover_name='country',                 
    log_x=True,                             # log scale: wealth is non-linear
    labels={'gdpPercap': 'GDP per Capita (USD, log scale)',
            'lifeExp': 'Life Expectancy',
            'continent': ''},
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_layout(
    title='GDP per capita vs life expectancy for 2002 and 2007',
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    yaxis=dict(gridcolor='#EEEEEE'),
    xaxis=dict(showgrid=True, gridcolor='#EEEEEE'),
    margin=dict(l=60, r=40, t=55, b=40), height=500
)
fig.update_traces(marker=dict(size=9, opacity=0.7))
fig.show()

## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [62]:
# Task 2
# YOUR CODE HERE
import math
Continent = random.choice(list(df["continent"].drop_duplicates()))
df1 = df.loc[df['year'] == 2007]
df2["year"] = df2["year"].astype("str")


df3 = df1.loc[df1['continent'] == Continent]
Country1 = random.choice(list(df3["country"].drop_duplicates()))
Country2 = random.choice(list(df3["country"].drop_duplicates()))
while Country2 == Country1:
    Country2 = random.choice(list(df3["country"].drop_duplicates()))
print(Country1)
x1=df3.loc[df3['country'] == Country1]['gdpPercap']
y1=df3.loc[df3['country'] == Country1]['lifeExp']
x2=df3.loc[df3['country'] == Country2]['gdpPercap']
y2=df3.loc[df3['country'] == Country2]['lifeExp']

Title = Country1 + ' and ' + Country2 + ': Big Countries in a BIG world'
 
color_map = {c: 'DarkBlue' if c == Continent else 'LightGrey' for c in df['continent'].unique()}

fig = px.scatter(
    data_frame=df, x='gdpPercap', y='lifeExp',
    size='pop',
    color='continent',
    hover_name='country',
    log_x=True,
    size_max=60,                            # cap bubble size so small countries still visible
    labels={'gdpPercap': 'GDP per Capita (log scale)', 'lifeExp': 'Life Expectancy', 'pop': 'Population', 'continent': ''},
    color_discrete_map=color_map,
    custom_data=['country', 'pop'],
    opacity=0.7
)

# Improve hover: show formatted population using HTML
fig.update_traces(
    hovertemplate='<b>%{hovertext}</b><br>'
                  'GDP per capita: $%{x:,.0f}<br>'
                  'Life expectancy: %{y:.1f} yrs<br>'
                  'Population: %{customdata[1]:,.0f}')

fig.update_layout(
    title= Title,
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    yaxis=dict(gridcolor='#EEEEEE'),
    xaxis=dict(gridcolor='#EEEEEE'),
    margin=dict(l=60, r=40, t=55, b=40),
    height=600, width=1150
)


x1 = math.log10(int(x1.values[0])) + 3
x2 = math.log10(int(x2.values[0])) + 3
y1 = int(y1.values[0])
y2 = int(y2.values[0])


for country, ax, ay in [(Country1, x1, y1), (Country2, x2, y2)]:
    row = df3[df3['country'] == country].iloc[0]
    fig.add_annotation(
        x=math.log10(row['gdpPercap']), y=row['lifeExp'],
        text=f'<b>{country}</b>', showarrow=True, arrowcolor='rgb(127, 60, 141)',
        arrowhead=1, ax=ax, ay=ay,
        font=dict(size=15, family='Arial', color='rgb(127, 60, 141)')
    )
    


fig.show()




Sierra Leone
